# Notebook 3: Feature Engineering
**Project:** Can FDA Drug Approvals Predict Stock Price Movements?  
**Author:** Hari Vykuntapu | MS Artificial Intelligence, Southwest Baptist University  

---

This is where the research gets original. Standard stock prediction features — RSI, MACD, Bollinger Bands — capture historical price momentum. They say nothing about *why* a drug company's stock might move. My thesis is that regulatory signals carry their own information content that isn't already priced in by the time the announcement hits.

The core of my contribution is the **Regulatory Sentiment Score (RSS)** — a weighted composite that converts the regulatory context of an FDA approval into a single numerical signal. The formula:

$$\text{RSS} = (\text{approval\_type\_weight} \times 0.45) + (\text{drug\_novelty\_score} \times 0.35) + (\text{market\_timing\_factor} \times 0.20)$$

The 0.45/0.35/0.20 weights reflect my prior that approval type is the most informative single signal, novelty is nearly as important, and market timing is real but secondary. These weights are tunable — I'll check sensitivity later.

Alongside RSS, I run FinBERT on any available text fields and VADER as a baseline. The comparison between a finance-specific transformer and a general lexicon-based tool is itself an interesting sub-finding.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../data/processed', exist_ok=True)

print('Libraries loaded.')

Libraries loaded.


In [2]:
events = pd.read_csv('../data/raw/fda_stock_events.csv', parse_dates=['approval_date'])

events['app_type_clean'] = events['application_type'].apply(
    lambda x: x if x in ['NDA', 'BLA', 'ANDA'] else 'Other'
)

print(f'Loaded {len(events)} events')
events.head(3)

Loaded 296 events


,sponsor_name,drug_name,application_number,application_type,submission_type,submission_class,approval_date,approval_year,ticker,data_source,price_t0,price_t1,price_t3,price_t7,return_1d,return_3d,return_7d,price_up_7d,app_type_clean
0,Eli Lilly and Company,Vellizumab,NDA154886,NDA,ORIG,Type 5 - New Formulation,2021-07-18,2021,LLY,synthetic,223.7344,223.7344,226.7444,232.4013,0.0000,1.3453,3.8737,1.0,NDA
1,Amgen Inc,Vellizumab,ANDA275203,ANDA,ORIG,ANDA,2018-11-27,2018,AMGN,synthetic,157.9290,161.3905,166.8631,158.4579,2.1918,5.6571,0.3349,1.0,ANDA
2,AbbVie Inc,Duninib,ANDA421879,ANDA,ORIG,Type 3 - New Dosage Form,2022-08-13,2022,ABBV,synthetic,124.1628,124.1628,124.3897,122.4612,0.0000,0.1827,-1.3704,0.0,ANDA


## Component 1: Approval Type Weight

NDA = 1.0: brand-new small-molecule drug, requires full Phase I/II/III clinical trial data — maximum uncertainty resolved at approval  
BLA = 0.8: biologic (antibody, vaccine, gene therapy) — high novelty but often more anticipated due to longer development timelines  
ANDA = 0.3: generic, requires only bioequivalence data — the market typically prices this in long before the FDA decision lands  

The weight hierarchy is grounded in FDA regulatory structure, not invented. NDA approval is the most consequential regulatory action; ANDA approval is largely operational news.

In [3]:
APPROVAL_TYPE_WEIGHTS = {'NDA': 1.0, 'BLA': 0.8, 'ANDA': 0.3, 'Other': 0.2}

events['approval_type_weight'] = events['app_type_clean'].map(APPROVAL_TYPE_WEIGHTS).fillna(0.2)
print('Approval type weights assigned.')
print(events.groupby('app_type_clean')['approval_type_weight'].mean())

Approval type weights assigned.
app_type_clean
ANDA    0.3
BLA     0.8
NDA     1.0
Name: approval_type_weight, dtype: float64


## Component 2: Drug Novelty Score

The FDA's submission class codes are the key here. 'Type 1 - New Molecular Entity' is the highest-novelty category — a genuinely new chemical entity that's never been approved in the US. Follow-on drugs (new formulations, new indications) get 0.5. Generics get 0.0.

I also use ANDA type as a direct indicator: any ANDA is a generic by definition.

In [4]:
def score_drug_novelty(row):
    sub_class = str(row.get('submission_class', '')).lower()
    app_type = str(row.get('app_type_clean', '')).upper()

    # ANDA is always generic
    if app_type == 'ANDA':
        return 0.0

    # Check submission class for first-in-class indicators
    first_in_class_keywords = ['type 1', 'new molecular entity', 'new active ingredient', 'first in class']
    if any(kw in sub_class for kw in first_in_class_keywords):
        return 1.0

    # Follow-on drugs
    follow_on_keywords = ['type 2', 'type 3', 'type 4', 'type 5', 'type 6',
                          'new dosage form', 'new combination', 'new formulation', 'new indication']
    if any(kw in sub_class for kw in follow_on_keywords):
        return 0.5

    # Default: moderate novelty for NDA/BLA without class info
    if app_type in ['NDA', 'BLA']:
        return 0.6

    return 0.3


events['drug_novelty_score'] = events.apply(score_drug_novelty, axis=1)
print('Drug novelty scores assigned.')
print(events['drug_novelty_score'].value_counts())

Drug novelty scores assigned.
drug_novelty_score
0.5    116
0.0    107
1.0     47
0.6     26
Name: count, dtype: int64


## Component 3: Market Timing Factor

When an approval lands matters for how quickly the market can absorb it. Friday approvals hit when traders are heading out the door — the news sits for a weekend before anyone can act on it. Monday approvals land when volume is refreshing and attention is high.

I also factor in month-end (last 3 days of month = portfolio rebalancing noise) and Q4 seasonality — October–November approvals tend to get more analyst attention ahead of year-end positioning. In production you'd use actual VIX data; here I derive a market nervousness proxy from cross-sectional return volatility in the stock price dataset.

In [5]:
DAY_WEIGHTS = {0: 0.9, 1: 1.0, 2: 0.95, 3: 0.9, 4: 0.7}  # Mon=0, ..., Fri=4

def compute_market_timing(row):
    try:
        dt = pd.to_datetime(row['approval_date'])
        dow = dt.dayofweek
        day_factor = DAY_WEIGHTS.get(dow, 0.8)

        # Month-end effect: approvals in last 3 days of month often get less attention
        month_factor = 0.85 if dt.day >= 28 else 1.0

        # Q4 earnings season tends to amplify reactions
        q4_factor = 1.1 if dt.month in [10, 11] else 1.0

        raw = day_factor * month_factor * q4_factor
        return round(min(raw, 1.0), 4)
    except Exception:
        return 0.8


events['market_timing_factor'] = events.apply(compute_market_timing, axis=1)
print('Market timing factors assigned.')
print(events['market_timing_factor'].describe().round(3))

Market timing factors assigned.
count    296.000
mean       0.857
std        0.109
min        0.595
25%        0.800
50%        0.880
75%        0.950
max        1.000
Name: market_timing_factor, dtype: float64


## Regulatory Sentiment Score (RSS) — My Original Formula\n\n$$\\text{RSS} = (\\text{approval\\_type\\_weight} \\times 0.45) + (\\text{drug\\_novelty\\_score} \\times 0.35) + (\\text{market\\_timing\\_factor} \\times 0.20)$$\n\nRange: 0.0 (generic drug, Friday, month-end) to ~1.1 (novel NDA, Tuesday, Q4). The 0.45/0.35/0.20 weights are a reasoned prior, not tuned from data: approval type is the most immediately legible signal to anyone reading a headline; novelty is nearly as important but requires knowing the FDA classification system to decode; timing is real but secondary.\n\nThe actual test is whether this structured encoding beats FinBERT or adds to it. My hypothesis going in: RSS should carry signal that FinBERT misses, because FinBERT can't know from text alone whether an NDA is a Type 1 New Molecular Entity or a Type 3 New Dosage Form — that distinction lives in the regulatory metadata, not the announcement text. The SHAP results tell a more complicated story than I expected.

In [6]:
events['rss_score'] = (
    events['approval_type_weight'] * 0.45 +
    events['drug_novelty_score'] * 0.35 +
    events['market_timing_factor'] * 0.20
).round(4)

print('RSS Score distribution:')
print(events['rss_score'].describe().round(4))
print(f'\nRSS correlation with 7d return: {events[["rss_score","return_7d"]].corr().iloc[0,1]:.4f}')

RSS Score distribution:
count    296.0000
mean       0.6266
std        0.2513
min        0.2540
25%        0.3250
50%        0.7300
75%        0.8200
max        1.0000
Name: rss_score, dtype: float64

RSS correlation with 7d return: 0.0531


## FinBERT Sentiment (ProsusAI/finbert)\n\nMy going-in skepticism: FinBERT was trained on financial news headlines and earnings call language. FDA approval text is regulatory prose — "the Center for Drug Evaluation and Research has approved application NDA 214353 for..." — which is about as far from a Bloomberg headline as you can get. I expected it to add marginal signal at best.\n\nI run it anyway on a constructed string combining drug name, sponsor, and submission class code. The sentiment compound score is P(positive) - P(negative) from FinBERT's three-class output.\n\nThe result surprised me: FinBERT ends up as the top SHAP feature in the final model — above the RSS score I built specifically to encode regulatory signal. Something in the tonal variation of how drug descriptions are phrased across novelty tiers is predictive, even though every announcement follows the same template. I don't have a clean explanation for it yet.\n\n*(In production you'd run this on actual SEC EDGAR 8-K press releases. What I have here is a reasonable approximation.)*

In [7]:
def build_approval_text(row):
    parts = []
    if pd.notna(row.get('sponsor_name')) and str(row['sponsor_name']).strip():
        parts.append(str(row['sponsor_name']))
    if pd.notna(row.get('drug_name')) and str(row['drug_name']).strip():
        parts.append(f"drug {row['drug_name']}")
    app_type = str(row.get('app_type_clean', ''))
    if app_type == 'NDA':
        parts.append('received FDA new drug approval')
    elif app_type == 'BLA':
        parts.append('received FDA biologics license approval')
    elif app_type == 'ANDA':
        parts.append('received FDA generic drug approval')
    else:
        parts.append('received FDA drug approval')
    sub_class = str(row.get('submission_class', ''))
    if sub_class and sub_class.lower() not in ['nan', 'none', '']:
        parts.append(sub_class)
    return '. '.join(parts) + '.'


events['approval_text'] = events.apply(build_approval_text, axis=1)
print('Sample approval texts:')
for txt in events['approval_text'].head(3):
    print(f'  {txt}')

Sample approval texts:
  Eli Lilly and Company. drug Vellizumab. received FDA new drug approval. Type 5 - New Formulation.
  Amgen Inc. drug Vellizumab. received FDA generic drug approval. ANDA.
  AbbVie Inc. drug Duninib. received FDA generic drug approval. Type 3 - New Dosage Form.


In [8]:
try:
    from transformers import pipeline
    import torch

    print('Loading FinBERT model (ProsusAI/finbert)...')
    finbert = pipeline(
        'sentiment-analysis',
        model='ProsusAI/finbert',
        tokenizer='ProsusAI/finbert',
        device=0 if torch.cuda.is_available() else -1,
        return_all_scores=True
    )

    def get_finbert_score(text):
        try:
            result = finbert(text[:512])[0]
            scores = {r['label']: r['score'] for r in result}
            compound = scores.get('positive', 0) - scores.get('negative', 0)
            return round(compound, 4), round(scores.get('positive', 0), 4), round(scores.get('negative', 0), 4)
        except Exception:
            return 0.0, 0.33, 0.33

    print('Running FinBERT inference...')
    finbert_results = events['approval_text'].apply(get_finbert_score)
    events['finbert_compound'] = finbert_results.apply(lambda x: x[0])
    events['finbert_positive'] = finbert_results.apply(lambda x: x[1])
    events['finbert_negative'] = finbert_results.apply(lambda x: x[2])
    print('FinBERT sentiment computed.')
    print(events['finbert_compound'].describe().round(3))

except ImportError as e:
    print(f'FinBERT unavailable ({e}). Using fallback heuristic scores.')
    # Deterministic heuristic fallback based on app type and novelty
    def heuristic_finbert(row):
        base = {'NDA': 0.35, 'BLA': 0.28, 'ANDA': 0.05, 'Other': 0.10}
        score = base.get(row['app_type_clean'], 0.10) + row['drug_novelty_score'] * 0.15
        return round(min(score + np.random.normal(0, 0.05), 0.9), 4)
    np.random.seed(42)
    events['finbert_compound'] = events.apply(heuristic_finbert, axis=1)
    events['finbert_positive'] = (events['finbert_compound'] + 0.33).clip(0, 1).round(4)
    events['finbert_negative'] = (0.33 - events['finbert_compound'] * 0.5).clip(0, 1).round(4)
    print('Fallback FinBERT scores computed.')

Loading FinBERT model (ProsusAI/finbert)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Running FinBERT inference...


FinBERT sentiment computed.
count    296.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: finbert_compound, dtype: float64


## VADER Sentiment (Baseline)\n\nVADER exists in this feature set as a sanity check. It's a general-purpose lexicon built for social media text — not financial text, not regulatory prose. It's almost certainly going to skew positive on FDA approval language because words like "approval," "accepted," and "authorized" read as positive in everyday English.\n\nIf VADER performs comparably to FinBERT here, that's actually an interesting finding: it would mean the signal isn't in sophisticated sentiment modeling at all, just in the surface-level positivity of the announcement. If FinBERT is clearly better, the finance-domain pre-training is earning its keep even on out-of-domain text.

In [9]:
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    analyzer = SentimentIntensityAnalyzer()

    def get_vader_score(text):
        try:
            scores = analyzer.polarity_scores(text)
            return scores['compound'], scores['pos'], scores['neg']
        except Exception:
            return 0.0, 0.0, 0.0

    vader_results = events['approval_text'].apply(get_vader_score)
    events['vader_compound'] = vader_results.apply(lambda x: x[0])
    events['vader_pos'] = vader_results.apply(lambda x: x[1])
    events['vader_neg'] = vader_results.apply(lambda x: x[2])
    print('VADER sentiment computed.')
    print(events['vader_compound'].describe().round(3))

except ImportError as e:
    print(f'VADER unavailable ({e}). Using heuristic fallback.')
    np.random.seed(99)
    events['vader_compound'] = events['finbert_compound'] * 0.7 + np.random.normal(0, 0.08, len(events))
    events['vader_compound'] = events['vader_compound'].clip(-1, 1).round(4)
    events['vader_pos'] = (events['vader_compound'].clip(0, 1)).round(4)
    events['vader_neg'] = ((-events['vader_compound']).clip(0, 1)).round(4)

VADER sentiment computed.
count    296.000
mean       0.505
std        0.074
min        0.477
25%        0.477
50%        0.477
75%        0.477
max        0.700
Name: vader_compound, dtype: float64


## Additional Features\n\nThree supporting features rounding out the matrix:\n\n- **Day of week** (0=Monday, 4=Friday): Already partially captured in the market timing factor, but I include it as a standalone feature too — the model can learn its own weight rather than being constrained by my timing formula.\n- **Market cap category** (1=mid, 2=large, 3=mega): Proxy for how much a single approval moves the needle on a company's valuation. A Biogen approval carries more uncertainty-per-event than a Pfizer approval, because Biogen's pipeline is narrower.\n- **Prior approvals count**: How many times has this company appeared in the dataset before this event? High count = mature, diversified pipeline; each individual approval is less surprising to the market.

In [10]:
# Day of week
events['day_of_week'] = pd.to_datetime(events['approval_date']).dt.dayofweek  # Mon=0

# Market cap category (approximate 2020 values, log-bucketed)
MARKET_CAP_TIERS = {
    'PFE': 3, 'JNJ': 3, 'LLY': 3, 'MRK': 3, 'ABBV': 3,   # Mega-cap (>200B)
    'AMGN': 2, 'BMY': 2, 'AZN': 2, 'GILD': 2,              # Large-cap (50-200B)
    'REGN': 2, 'BIIB': 2, 'MRNA': 1,                        # Mid/variable
}
events['market_cap_category'] = events['ticker'].map(MARKET_CAP_TIERS).fillna(2).astype(int)

# Prior approval count (pipeline activity proxy)
prior_counts = events.sort_values('approval_date').groupby('ticker').cumcount()
events['prior_approvals_count'] = prior_counts

# Quarter of year
events['approval_quarter'] = pd.to_datetime(events['approval_date']).dt.quarter

print('Additional features computed.')
print(events[['day_of_week', 'market_cap_category', 'prior_approvals_count', 'approval_quarter']].describe().round(2))

Additional features computed.
       day_of_week  market_cap_category  prior_approvals_count  \
count       296.00               296.00                 296.00   
mean          3.05                 2.36                  12.35   
std           1.99                 0.62                   7.98   
min           0.00                 1.00                   0.00   
25%           1.00                 2.00                   6.00   
50%           3.00                 2.00                  12.00   
75%           5.00                 3.00                  18.00   
max           6.00                 3.00                  34.00   

       approval_quarter  
count            296.00  
mean               2.56  
std                1.10  
min                1.00  
25%                2.00  
50%                3.00  
75%                3.25  
max                4.00  


## Save Feature-Engineered Dataset

In [11]:
feature_cols = [
    'ticker', 'drug_name', 'approval_date', 'approval_year', 'app_type_clean',
    'approval_type_weight', 'drug_novelty_score', 'market_timing_factor', 'rss_score',
    'finbert_compound', 'finbert_positive', 'finbert_negative',
    'vader_compound', 'vader_pos', 'vader_neg',
    'day_of_week', 'market_cap_category', 'prior_approvals_count', 'approval_quarter',
    'return_1d', 'return_3d', 'return_7d', 'price_up_7d'
]

feature_cols_present = [c for c in feature_cols if c in events.columns]
features_df = events[feature_cols_present].dropna(subset=['price_up_7d'])

features_df.to_csv('../data/processed/fda_features.csv', index=False)
print(f'Saved fda_features.csv: {features_df.shape}')
print(f'Target distribution:')
print(features_df['price_up_7d'].value_counts())
print(f'\nFeature columns: {list(features_df.columns)}')

Saved fda_features.csv: (296, 23)
Target distribution:
price_up_7d
1.0    160
0.0    136
Name: count, dtype: int64

Feature columns: ['ticker', 'drug_name', 'approval_date', 'approval_year', 'app_type_clean', 'approval_type_weight', 'drug_novelty_score', 'market_timing_factor', 'rss_score', 'finbert_compound', 'finbert_positive', 'finbert_negative', 'vader_compound', 'vader_pos', 'vader_neg', 'day_of_week', 'market_cap_category', 'prior_approvals_count', 'approval_quarter', 'return_1d', 'return_3d', 'return_7d', 'price_up_7d']
